In [ ]:
import numpy as np
import pandas as pd

import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    LSTM,
    RepeatVector,
    TimeDistributed,
    Dense
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import StandardScaler

In [ ]:
if devices := tf.config.list_physical_devices("GPU"):
    print(f"Running on {devices}")
else:
    print("Running on CPU")

In [ ]:
# load API key
try:
    from google.colab import userdata
    IN_COLAB = True
except:
    IN_COLAB = False


if IN_COLAB:
    %pip install -q zarr xarray fsspec aiohttp earthkit
    cdsapi_key = userdata.get("CDS-API-KEY")
else:
    import os

    %load_ext dotenv
    %dotenv

    cdsapi_key = os.getenv("CDS-API-KEY")


In [ ]:
import xarray as xr

# https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land

# Choose chunking approach
# "geo" for Geo-chunked stores for access optimised along the time dimension (e.g. time-series at a point)
# "time" for Time-chunked stores for access optimised in spatial dimensions (e.g. a global map)
chunks = "geo" # or "time"

# Soil temperature
soil_temperature_url = f"https://arco.datastores.ecmwf.int/cadl-arco-{chunks}-006/arco/reanalysis_era5_land/sfc-soil-temperature/{chunks}Chunked.zarr"
# 2m temperature
var_2m_temperature_url = f"https://arco.datastores.ecmwf.int/cadl-arco-{chunks}-007/arco/reanalysis_era5_land/sfc-2m-temperature/{chunks}Chunked.zarr"
# Soil water
soil_water_url = f"https://arco.datastores.ecmwf.int/cadl-arco-{chunks}-005/arco/reanalysis_era5_land/sfc-soil-water/{chunks}Chunked.zarr"
# Radiation and heat
radiation_and_heat_url = f"https://arco.datastores.ecmwf.int/cadl-arco-{chunks}-010/arco/reanalysis_era5_land/sfc-radiation-heat/{chunks}Chunked.zarr"
# Snow
snow_url = f"https://arco.datastores.ecmwf.int/cadl-arco-{chunks}-030/arco/reanalysis_era5_land/sfc-snow/{chunks}Chunked.zarr"
# Wind
wind_url = f"https://arco.datastores.ecmwf.int/cadl-arco-{chunks}-008/arco/reanalysis_era5_land/sfc-wind/{chunks}Chunked.zarr"
# Pressure and precipitation
pressure_and_precipitation_url = f"https://arco.datastores.ecmwf.int/cadl-arco-{chunks}-009/arco/reanalysis_era5_land/sfc-pressure-precipitation/{chunks}Chunked.zarr"
# Skin temperature
skin_temperature_url = f"https://arco.datastores.ecmwf.int/cadl-arco-{chunks}-043/arco/reanalysis_era5_land/sfc-skin-temperature/{chunks}Chunked.zarr"

datasets = [
    xr.open_zarr(
        var_2m_temperature_url,
        consolidated=True,
        storage_options={
            "headers": {"Authorization": f"Bearer {cdsapi_key}"}
        }
    ),
    xr.open_zarr(
        wind_url,
        consolidated=True,
        storage_options={
            "headers": {"Authorization": f"Bearer {cdsapi_key}"}
        }
    ),
    xr.open_zarr(
        pressure_and_precipitation_url,
        consolidated=True,
        storage_options={
            "headers": {"Authorization": f"Bearer {cdsapi_key}"}
        }
    )
]

# only use data from Frankfurt am Main
ds = xr.merge(datasets)
# ds = ds_wind
ds = ds.sel(latitude=50.13, longitude=8.69, method="nearest")

# Inspect the variables
ds.load()
ds

In [ ]:
# # load dataset

# import xarray as xr

# # Geo-chunked surface levels data (optimised for time-series at a single location)
# geochunked_sfc_url = "https://arco.datastores.ecmwf.int/cadl-arco-geo-002/arco/reanalysis_era5_single_levels/sfc/geoChunked.zarr"

# # Open the zarr store with xarray, users must insert their API key where indicated.
# ds = xr.open_zarr(
#     geochunked_sfc_url,
#     consolidated=True,
#      storage_options = {
#         "headers": {"Authorization": f"Bearer {cdsapi_key}"}
#     }
# )

# # only use data from Frankfurt am Main
# ds = ds.sel(latitude=50.7, longitude=8.41, method="nearest")

# # ignore the first day, where tp is NaN
# ds = ds.sel(time=slice("1940-01-02", None))

# # select variables
# variables = {
#     "t2m": "2m temperature in Kelvin",
#     "d2m": "2m dewpoint in Kelvin",
#     "msl": "mean sea level pressure in Pascal", # surface pressure better?
#     "u10": "10m wind eastward component in metre per second",
#     "v10": "10m wind northward component in metre per second",
#     "tcc": "total cloud cover in fractions between 0 and 1",
#     "tp":  "total precipitation in metre"
# }
# ds = ds[variables.keys()]

# # Inspect the variables
# ds.load()
# ds

In [ ]:
# plot raw example data

from earthkit import plots as ekp
from earthkit import transforms as ekt

import warnings
warnings.filterwarnings(
    "ignore",
    message="TimeSeries is an experimental new feature*"
)

def plot_variable(variable_name, title):
    time = slice("2025-01-01", "2026-01-01")

    plot_data_hourly = ds[variable_name].sel(time=time)
    plot_data_monthly = ekt.temporal.monthly_mean(plot_data_hourly)

    chart = ekp.TimeSeries()

    chart.line(plot_data_hourly, label="Hourly")
    chart.line(plot_data_monthly, label="Monthly mean")

    chart.title(title)
    chart.legend()

    chart.show()


plot_variable("d2m", "2m dewpoint in Kelvin")
plot_variable("t2m", "2m temperature in Kelvin")
plot_variable("u10", "10m wind eastward component in metre per second")
plot_variable("v10", "10m wind northward component in metre per second")
plot_variable("sp", "surface pressure in Pascal")
plot_variable("tp", "total precipitation in metre")

In [ ]:
# move to pandas Dataframe
df = ds.to_dataframe().drop(columns=["latitude", "longitude"])
df

In [ ]:
# split the data chonologically
n = len(df)

train_end = int(n * 0.7)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end]
val_df   = df.iloc[train_end:val_end]
test_df  = df.iloc[val_end:]

In [ ]:
# scale data
scaler = StandardScaler()

train_scaled = scaler.fit_transform(train_df)
val_scaled = scaler.transform(val_df)
test_scaled = scaler.transform(test_df)

In [ ]:
# create encoder-decoder windows
INPUT_LEN = 168      # 7 days
OUTPUT_LEN = 24      # next 24 hours


def create_sequences(data, input_len, output_len):

    X = []
    y = []

    total = len(data)

    for i in range(total - input_len - output_len + 1):

        encoder = data[i : i + input_len]

        decoder = data[
            i + input_len :
            i + input_len + output_len
        ]

        X.append(encoder)
        y.append(decoder)

    return np.array(X), np.array(y)

X_train, y_train = create_sequences(
    train_scaled,
    INPUT_LEN,
    OUTPUT_LEN,
)

X_val, y_val = create_sequences(
    val_scaled,
    INPUT_LEN,
    OUTPUT_LEN,
)

X_test, y_test = create_sequences(
    test_scaled,
    INPUT_LEN,
    OUTPUT_LEN,
)

X_train.shape, y_train.shape

In [ ]:
# define model
n_features = X_train.shape[2]

inputs = Input(shape=(INPUT_LEN, n_features))

# Encoder
encoder = LSTM(
    128,
    activation="tanh"
)(inputs)

# Repeat context vector
decoder_input = RepeatVector(OUTPUT_LEN)(encoder)

# Decoder
decoder = LSTM(
    128,
    activation="tanh",
    return_sequences=True
)(decoder_input)

outputs = TimeDistributed(
    Dense(n_features)
)(decoder)

model = Model(inputs, outputs)

model.summary()

In [ ]:
# compile model
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

In [ ]:
print(np.isnan(X_train).any())
print(np.isnan(y_train).any())

print(np.max(X_train))
print(np.min(X_train))

In [ ]:
# train model

early_stop = EarlyStopping(
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
    shuffle=False
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(history.history["loss"], label="Training Loss")
if "val_loss" in history.history:
    plt.plot(history.history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))


plt.plot(history.history["mae"], label="Training MAE")
if "val_mae" in history.history:
    plt.plot(history.history["val_mae"], label="Validation MAE")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()


print("Keys:", history.history.keys())

In [ ]:
# predict
pred = model.predict(X_test)

print(pred.shape)

In [ ]:
# rescale features
pred_original = scaler.inverse_transform(
    pred.reshape(-1, n_features)
).reshape(pred.shape)

y_original = scaler.inverse_transform(
    y_test.reshape(-1, n_features)
).reshape(y_test.shape)